# E2：多目标优化 —— 动力 vs 经济权衡（pymoo）

## 实验目标
- 理解 Pareto 前沿
- 用 pymoo 做双目标优化

## 通过判据
```python
assert len(pareto_front) >= 5
```

In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np
print('✓ pymoo 导入成功')

In [ ]:
class EVDesign(Problem):
    """电动车设计：最小化 0-100km/h 时间 AND 百公里电耗"""
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, n_constr=0,
                         xl=np.array([50, 1000]), xu=np.array([200, 2500]))
        # x[0]: 电机功率 kW, x[1]: 车重 kg
    def _evaluate(self, x, out, *args, **kwargs):
        power, weight = x[:, 0], x[:, 1]
        # 简化模型：加速时间 ~ weight/power，能耗 ~ weight/sqrt(power)
        t_0100 = 2.5 * weight / power
        energy = 8 + 0.004 * weight / np.sqrt(power / 100)
        out['F'] = np.column_stack([t_0100, energy])

problem = EVDesign()
res = minimize(problem, NSGA2(pop_size=40), ('n_gen', 30), verbose=False)
pareto = res.F
print(f'Pareto 解数: {len(pareto)}')
assert len(pareto) >= 5, 'Pareto 解太少!'
print('✅ 评测通过！')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(pareto[:, 0], pareto[:, 1], c='blue', alpha=0.6)
plt.xlabel('0-100km/h 时间 (s)'); plt.ylabel('百公里电耗 (kWh)')
plt.title('Pareto 前沿：动力 vs 经济')
plt.grid(True, alpha=0.3); plt.show()

## 🏆 挑战任务
1. 加第三个目标：成本，跑 3 目标优化。
2. 把 Pareto 解的电机功率映射到 FASTSim 真实车型，验证简化模型的准确性。